# Modelo Híbrido DenseNet121 + MLP


## Objetivo do Script_09 no Escopo Silver KNN

O script_09, em sua versão original, itera sobre os 5 datasets treinando um modelo por dataset. Na configuração
Silver KNN deste roteiro, o loop é simplificado para treinar apenas 1 modelo — o SILVER_KNN — avaliado no test
set canônico do GOLD. Isso reduz o tempo total de execução em aproximadamente 80%.
5.3 Adaptação do CONFIG para Rodar Apenas Silver KNN

```python
# Em CONFIG['datasets'] (script_09_modelo_hibrido.py):
# Manter apenas o SILVER_KNN no dicionário:
'datasets': {
'GOLD' : {'file': '../csv/ecg_gold_completo_classified.csv',
'n': 3013, 'imputacao_pct': 0.0, 'metodo': 'Sem imputacao'},
'SILVER_KNN': {'file': '../csv/ecg_silver_knn_imputado_classified.csv',
'n': 3481, 'imputacao_pct': 1.67, 'metodo': 'KNN'},
# Remover ou comentar BRONZE_HYBRID, BRONZE_MICE, BRONZE_KNN
},
# ATENÇÃO: GOLD deve permanecer no dicionário pois gerar_analise_comparativa()
# faz referência a todos_metricas['GOLD'] para calcular Delta_GOLD.
# Ou adapte a função para calcular delta em relação ao próprio SILVER_KNN.
```

> Alternativa simplificada:
> 
> Se o objetivo é apenas treinar e avaliar o SILVER KNN sem comparação com GOLD, pode-se também remover o GOLD do dicionário e simplificar a função gerar_analise_comparativa() para não calcular Delta_GOLD. Nesse caso, os resultados serão apresentados apenas para o SILVER KNN.


## Arquitetura do Modelo Híbrido

A arquitetura HybridECGClassifier integra dois branches especializados fundidos por concatenação:

| Componente | Input → Output                 | Descrição                                                                                                   |
| ---------- | ------------------------------ | ----------------------------------------------------------------------------------------------------------- |
| Branch CNN | `[B, 1, 272, 512] → [B, 1024]` | DenseNet121 pré-treinado em ImageNet, com `conv0` adaptado para 1 canal usando a média dos pesos originais. |
| Branch MLP | `[B, 14] → [B, 32]`            | Rede densa composta por `Linear(14→64)` + `ReLU` + `Dropout(0.4)` + `Linear(64→32)`.                        |
| Fusão      | `[B, 1056] → [B, 2]`           | `CONCAT([1024, 32])` seguido de `FC(1056→256→128→2)` + `Dropout` + `Softmax`.                               |




## Estratégia de Treinamento em 2 Fases

### Fase 1 — Warm-up (10 épocas, CNN congelada)

- CNN (DenseNet121) completamente congelada — apenas MLP e camadas de fusão treinadas
- LR único: 1e-3 (otimizador Adam)
- Objetivo: inicializar os pesos da fusão sem destruir representações ImageNet da CNN


### Fase 2 — Fine-tuning (40 épocas, rede completa)

- Todos os parâmetros liberados para treinamento
- LRs diferenciados: CNN: 1e-5 | MLP: 1e-4 | Fusão: 1e-4
- ReduceLROnPlateau: fator 0.5, paciência 5 épocas no val_loss
- Early stopping: paciência 10 épocas no val_AUC
- Gradient clipping: grad_clip=1.0 para estabilidade numérica
- Class weights: calculados sobre y_train do SILVER KNN via compute_class_weight


## Métricas de Avaliação

O modelo é avaliado no test set canônico do GOLD (452 registros). As métricas calculadas são:
- AUC-ROC: principal métrica — meta >= 0,95
- Accuracy: acurácia geral no test set
- F1 Macro: média dos F1 de ambas as classes
- F1 NORMAL (classe 0) e F1 ANORMAL (classe 1): análise por classe
- Baseline de referência: XGBoost tabular puro — AUC = 0,928


## Execução

- Garantir que o test set canônico do GOLD está disponível
```bash
ls splits/test_indices.npy # deve existir e conter 452 índices GOLD
```

> Tempo estimado: 30–90 minutos (CPU) | 10–25 minutos (GPU NVIDIA com CUDA). Com SILVER KNN e apenas
> 1 modelo (sem os 4 datasets adicionais), o tempo é reduzido ~80% em relação ao script original.


## Artefatos Gerados

| Artefato                     | Local                    | Conteúdo                                                                   |
| ---------------------------- | ------------------------ | -------------------------------------------------------------------------- |
| `modelo_SILVER_KNN_final.pt` | `checkpoints/`           | Pesos finais do modelo, métricas, configurações e scaler.                  |
| `historico_SILVER_KNN.json`  | `resultados_e_metricas/` | Histórico de treino contendo loss e AUC por época (warm-up + fine-tuning). |
| `comparative_results.csv`    | `resultados_e_metricas/` | Tabela comparativa com métricas do(s) dataset(s) treinado(s).              |
| `comparative_roc.png`        | `plots_comparativos/`    | Curva ROC do modelo SILVER KNN comparada ao baseline XGBoost.              |
| `comparative_auc_barras.png` | `plots_comparativos/`    | Gráfico de barras da AUC-ROC com linha de meta em `0.95`.                  |
| `script_09_MAIN_*.log`       | `resultados_e_metricas/` | Log completo de execução contendo timestamps e métricas por época.         |


# Config

In [8]:
import os
import sys
import time
import json
import logging
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import models
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score,
    classification_report, roc_curve
)

from ipynb.fs.full.script_08_pytorch_dataset import (
    ECGMultimodalDataset, get_train_transform, get_eval_transform
)

CONFIG = {
    # Paths
    'image_dir': '../image_tracings',
    'splits_dir': './splits',
    'results_dir': '../resultados_e_metricas/script_08_pytorch_dataset',
    'checkpoints_dir': '../checkpoints',

    # Cinco datasets do estudo comparativo
    'datasets': {
        'GOLD': {'file': '../csv/ecg_gold_completo_classified.csv', 'n': 3013, 'imputacao_pct': 0.0,   'metodo': 'Sem imputacao'},
        'SILVER_KNN': {'file': '../csv/ecg_silver_knn_imputado_classified.csv', 'n': 3481, 'imputacao_pct': 1.67,  'metodo': 'KNN'},
        # 'BRONZE_HYBRID': {'file': '../csv/ecg_bronze_hybrid_imputado_classified.csv', 'n': 3664, 'imputacao_pct': 3.0,   'metodo': 'Hibrido KNN+MICE'},
        # 'BRONZE_MICE': {'file': '../csv/ecg_bronze_mice_imputado_classified.csv', 'n': 3664, 'imputacao_pct': 3.0,   'metodo': 'MICE'},
        # 'BRONZE_KNN': {'file': '../csv/ecg_bronze_knn_imputado_classified.csv', 'n': 3664, 'imputacao_pct': 3.0,   'metodo': 'KNN'},
    },

    # Colunas
    'param_cols': [
        'HR', 'Pd', 'PR', 'QRS_Dur', 'QT', 'QTC',
        'P_axis', 'QRS_axis', 'T_axis',
        'RV5', 'SV1', 'RV5_SV1_sum', 'RV6', 'SV2'
    ],
    'label_col'    : 'classificacao',
    'filename_col' : 'filename',

    # Imagem]
    'img_resize'    : (272, 512),
    'img_channels'  : 1,
    'normalize_mean': [0.5],
    'normalize_std' : [0.5],

    # Data augmentation
    'aug_translate'  : 0.02,
    'aug_scale_min'  : 0.98,
    'aug_scale_max'  : 1.02,
    'aug_brightness' : 0.1,
    'aug_contrast'   : 0.1,

    # Arquitetura
    'cnn_features' : 1024,
    'mlp_out'      : 32,
    'fusion_dim'   : 1056,
    'n_classes'    : 2,
    'dropout'      : 0.4,

    # Treinamento
    'batch_size'  : 32,
    'num_workers' : 0,
    'pin_memory'  : False,
    'random_seed' : 42,

    # Fase 1 — Warm-up (CNN congelada)
    'warmup_epochs' : 10,
    'warmup_lr'     : 1e-3,

    # Fase 2 — Fine-tuning (rede completa, LRs diferenciados)
    'finetune_epochs' : 40,
    'lr_cnn'          : 1e-5,
    'lr_mlp'          : 1e-4,
    'lr_fusion'       : 1e-4,

    # Regularizacao e controle
    'weight_decay'        : 1e-4,
    'lr_patience'         : 5,
    'lr_factor'           : 0.5,
    'early_stop_patience' : 10,
    'grad_clip'           : 1.0,

    # Limiar H0: Delta AUC-ROC < 2% => robusto
    'delta_threshold' : 0.02,
}

# Configure logging

In [ ]:
def configurar_logging(results_dir, tag='MAIN'):
    os.makedirs(results_dir, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    log_path = os.path.join(results_dir, f'script_09_{tag}_{ts}.log')
    logger = logging.getLogger(f'rescue2_e7_{tag}')
    logger.setLevel(logging.INFO)
    if not logger.handlers:
        fmt = logging.Formatter('%(asctime)s  %(levelname)-8s  %(message)s',
                                datefmt='%H:%M:%S')
        fh = logging.FileHandler(log_path, encoding='utf-8')
        ch = logging.StreamHandler(sys.stdout)
        fh.setFormatter(fmt); ch.setFormatter(fmt)
        logger.addHandler(fh); logger.addHandler(ch)
    return logger

# Hybrid model -> DenseNet121 + MLP

In [ ]:
class HybridECGClassifier(nn.Module):
    def __init__(self, config):
        super().__init__()
        dropout = config['dropout']


In [17]:
import numpy as np
import pandas as pd

gold_test_idx = np.load('./splits/gold_test_path.npy')
df_gold = pd.read_csv('../csv/ecg_gold_completo_classified.csv')

test_filenames = df_gold.iloc[gold_test_idx]['filename'].values
print(f"Total test canônico: {len(test_filenames)}")
print(f"Primeiros 5 filenames: {test_filenames[:5]}")


df_silver = pd.read_csv('../csv/ecg_silver_knn_imputado_classified.csv')
encontrados = df_silver['filename'].isin(test_filenames).sum()
print(f"Filenames encontrados no SILVER: {encontrados}/{len(test_filenames)}")



Total test canônico: 452
Primeiros 5 filenames: <StringArray>
['image_data_0539.png', 'image_data_0907.png', 'image_data_2952.png',
 'image_data_6159.png', 'image_data_0921.png']
Length: 5, dtype: str
Filenames encontrados no SILVER: 452/452
